# Table of Contents
- [Description](#description)
- [Building SOC 2018 to O*NET 2019 Mapping](#building-soc-2018-to-onet-2019-mapping)
    - [Observations: SOC 2018 to O*NET-2019](#observations-soc-2018-to-onet-2019)
- [Building SOC 2010 to SOC 2018 Mapping](#building-soc-2010-to-soc-2018-mapping)
    - [Observations: Building SOC 2010 to SOC 2018 Mapping](#observations-building-soc-2010-to-soc-2018-mapping)
    - [Note](#note)
- [Building SOC 2010 to O*NET 2019 Crosswalk](#building-soc-2010-to-onet-2019-crosswalk)
- [Building 2010 Census Code to 2010 SOC Mapping](#building-2010-census-code-to-2010-soc-mapping)
    - [Observations: Building 2010 Census Code to 2010 SOC Mapping](#observations-building-2010-census-code-to-2010-soc-mapping)
- [Building 2010 Census Code to O*NET-SOC 2019 Crosswalk](#building-2010-census-code-to-onet-soc-2019-crosswalk)
- [ISCO 2008 to SOC 2010 Mapping](#isco-2008-to-soc-2010-mapping)
- [Building ISCO 2008 to O*NET-SOC 2019 Crosswalk](#building-isco-2008-to-onet-soc-2019-crosswalk)

# Description
The purpose of this notebook is to build the CSV that provides a mapping from each GSS Occupation Code (based on the SOC 2010 Census codes) to an O*NET-SOC 2019 Occupation Code.

Using the following crosswalks in `/data/crosswalks/`:
- `census_soc_2010_crosswalk.csv`
- `census2010_to_onet2019.csv`
- `onet2019_to_soc2018_crosswalk.csv`
- `soc2010_to_soc2018_crosswalk.csv`

This notebook constructs a crosswalk between GSS occupation codes (which are equivalent to the 2010 Census occupation codes) and O*NET-SOC 2019 occupation codes.

The resulting CSV file is stored under `/data/crosswalks/census2010_to_onet2019.csv`

In the case where one code maps to multiple other codes, tiebreaks will be determined using the following methods:
1. Exact job title matching
2. Bureau of Labor Statistics (BLS) historical data on total employment, to maximize coverage
3. Manual matching

# Building SOC 2018 to O*NET-2019 mapping

In [1]:
import pandas as pd
import csv

data_dir = "../../data/"
crosswalk_dir = data_dir + "crosswalks/"

# Read data into DataFrames
df_gss_soc = pd.read_csv(crosswalk_dir + "census_soc_2010_crosswalk.csv")
df_onet2019_soc2018 = pd.read_csv(crosswalk_dir + "onet2019_to_soc2018_crosswalk.csv")
df_soc2010_soc2018 = pd.read_csv(crosswalk_dir + "soc2010_to_soc2018_crosswalk.csv")

In [2]:
df_onet2019_soc2018.head()

,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,11-1011.00,Chief Executives,11-1011,Chief Executives
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
2,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
3,11-1031.00,Legislators,11-1031,Legislators
4,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers


In [3]:
df_onet2019_soc2018["O*NET-SOC 2019 Code"].duplicated().sum()

np.int64(0)

In [4]:
ambiguous_mask = df_onet2019_soc2018["2018 SOC Code"].duplicated(keep=False)
ambiguous_mask.sum()

np.int64(225)

### Observations: SOC 2018 to O*NET-2019
There are 149 ambiguous mappings (where one SOC 2018 code maps to many O*NET-SOC 2019 codes).

This will be handled by checking that for every ambiguous mapping, there is a "general" O*NET-SOC 2019 code (i.e. a code that ends with `.00`) that corresponds to each respective ambiguous SOC 2018 code. 

In [5]:
# Filter into two DataFrames: 
# 1. Ambiguous, which indicates O*NET-SOC 2019 maps to multiple 2018 SOC
# 2. Unambiguous, which indicates each O*NET-SOC 2019 maps to one 2018 SOC
df_onet_soc2018_unambiguous = df_onet2019_soc2018[~ambiguous_mask]
df_onet_soc2018_ambiguous = df_onet2019_soc2018[ambiguous_mask]

df_onet_soc2018_ambiguous

,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,11-1011.00,Chief Executives,11-1011,Chief Executives
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
10,11-3013.00,Facilities Managers,11-3013,Facilities Managers
11,11-3013.01,Security Managers,11-3013,Facilities Managers
13,11-3031.00,Financial Managers,11-3031,Financial Managers
...,...,...,...,...
975,53-6051.00,Transportation Inspectors,53-6051,Transportation Inspectors
976,53-6051.01,Aviation Inspectors,53-6051,Transportation Inspectors
977,53-6051.07,"Transportation Vehicle, Equipment and Systems ...",53-6051,Transportation Inspectors
986,53-7062.00,"Laborers and Freight, Stock, and Material Move...",53-7062,"Laborers and Freight, Stock, and Material Move..."


In [6]:
# Check every ambiguous SOC 2018 code mapping has general .00 code
df_onet_occ = pd.read_csv(data_dir + "onet_misc/occupation_data.csv")
onet2019_codes = set(df_onet_occ["O*NET-SOC Code"])

onet2019_to_soc2018 = {onet:soc for onet,soc 
                       in zip(df_onet_soc2018_unambiguous["O*NET-SOC 2019 Code"], 
                              df_onet_soc2018_unambiguous["2018 SOC Code"])}

# For every general code, add to our dictionary as 
    # [general_code:soc_code]
for idx,row in df_onet_soc2018_ambiguous.iterrows():
    soc_code = row["2018 SOC Code"]
    general_code = soc_code + ".00"
    if general_code in onet2019_to_soc2018:
        continue
    elif general_code in onet2019_codes:
        # This theoretically shouldn't happen
        onet2019_to_soc2018[general_code] = soc_code

if len(onet2019_to_soc2018) != df_onet2019_soc2018["2018 SOC Code"].nunique():
    print("[WARNING] We have not covered all valid SOC 2018 Codes")
else:
    print("[INFO] All SOC 2018 Codes have a corresponding O*NET 2019 Code")

# Flip our dictionary. This will be dict use for GSS to O*NET-2019 mapping
soc2018_to_onet2019 = {soc:onet for onet,soc in onet2019_to_soc2018.items()}

[INFO] All SOC 2018 Codes have a corresponding O*NET 2019 Code


# Building SOC 2010 to SOC 2018 Mapping

In [7]:
df_soc2010_soc2018.head()

,2010 SOC Code,2010 SOC Title,2018 SOC Code,2018 SOC Title
0,11-1011,Chief Executives,11-1011,Chief Executives
1,11-1021,General and Operations Managers,11-1021,General and Operations Managers
2,11-1031,Legislators,11-1031,Legislators
3,11-2011,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers
4,11-2021,Marketing Managers,11-2021,Marketing Managers


In [8]:
ambiguous_mask = df_soc2010_soc2018["2010 SOC Code"].duplicated(keep=False)
ambiguous_mask.sum()

np.int64(102)

### Observations: Building SOC 2010 to SOC 2018 Mapping
`df_soc2010_soc2018["2010 SOC Code"].duplicated().sum()` yields 60, meaning there are 60 codes from the SOC 2010 batch that map onto multiple 2018 SOC Codes. 

This will be handled by constructing two dictionaries: an `unambiguous` dictionary, containing mappings from SOC 2010 codes to SOC 2018 codes that are unambiguous (one to one), and an `ambiguous` dictionary, containing mappings from SOC 2010 codes to multiple SOC 2018 codes (one to many).

The "true" mappings for the `ambiguous` dictionary will be determined through a `tiebreaker()` method, where we pull from BLS data to determine which job code employed more people in the year of 2018 (for better coverage).

In [9]:
# Load BLS data for 2021
df_bls = pd.read_excel(crosswalk_dir + "bls_2021.xlsx")
print(f"[INFO] Columns are: {df_bls.columns}")

df_bls.head()

[INFO] Columns are: Index(['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE',
       'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP',
       'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT',
       'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN',
       'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75',
       'A_PCT90', 'ANNUAL', 'HOURLY'],
      dtype='str')


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,22,32.98,49.43,23980,29950,45760,68590,102810,NaN,NaN
1,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,49.25,77.39,#,47860,74710,102450,160960,#,NaN,NaN
2,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1000,Top Executives,...,47.46,77.18,#,41260,60900,98720,160540,#,NaN,NaN
3,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1010,Chief Executives,...,86.31,#,#,60300,111080,179520,#,#,NaN,NaN
4,99,U.S.,1,US,000000,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,86.31,#,#,60300,111080,179520,#,#,NaN,NaN


In [10]:
# Filter df_bls to the columns of interest since full DataFrame is large
# Convert TOT_EMP to integer
cols = ["OCC_CODE", "TOT_EMP"]
df_bls = df_bls[cols]
df_bls["TOT_EMP"] = pd.to_numeric(df_bls["TOT_EMP"], errors="coerce").astype("Int64")

In [11]:
# Get our ambiguous and unambiguous DataFrames
# Similar to before:
    # 1. Ambiguous is one SOC 2010 code to many SOC 2018 codes
    # 2. Unambiguous is one SOC 2010 code to one SOC 2018 codes
df_soc2010_soc2018_ambiguous = df_soc2010_soc2018[ambiguous_mask]
df_soc2010_soc2018_unambiguous = df_soc2010_soc2018[~ambiguous_mask]

In [12]:
# Build temp overall dictionary:
temp_soc2010_to_soc2018 = {}
for idx, row in df_soc2010_soc2018.iterrows():
    soc2010 = row["2010 SOC Code"]
    soc2019 = row["2018 SOC Code"]
    temp_soc2010_to_soc2018.setdefault(soc2010, []).append(soc2019)

In [13]:
# Build our disjoint unambiguous and ambiguous dictionaries
soc2010_to_soc2018 = {}
ambiguous_soc2010_to_soc2018 = {}
for soc2010, candidates in temp_soc2010_to_soc2018.items():
    if len(candidates) == 1:
        soc2010_to_soc2018[soc2010] = candidates[0]
    else:
        ambiguous_soc2010_to_soc2018[soc2010] = candidates

In [14]:
def tiebreaker(candidates, df_bls, occ_code="OCC_CODE", tot_emp="TOT_EMP"):
    '''Description: Tiebreaker, returns candidate with greatest total employed'''
    totals = (
        df_bls[df_bls[occ_code].isin(candidates)]
        .groupby(occ_code)[tot_emp]
        .sum()
        .reindex(candidates, fill_value=0)  # preserve candidates' order, 0 for missing ones
    )

    if totals.max() <= 0:
        return None

    return totals.idxmax()

# For SOC 2010 codes that are one to many, use tiebreaker() for best candidate
for soc2010, candidates in ambiguous_soc2010_to_soc2018.items():
    soc2018 = tiebreaker(candidates, df_bls)
    soc2010_to_soc2018[soc2010] = soc2018

In [15]:
# Check that we have good coverage after
full_coverage = (len(soc2010_to_soc2018) 
                 == df_soc2010_soc2018["2010 SOC Code"].nunique())

if not full_coverage:
    print("[WARNING] We have not covered all SOC 2010 Codes")
else:
    print("[INFO] All SOC 2010 Codes have an entry")

[INFO] All SOC 2010 Codes have an entry


In [16]:
# Now check if there are any 2010 SOC codes with entry but no BLS coverage
    # This might be because BLS did not collect data on that certain occupation
# SOC 2010 to SOC 2018 candidate map:
no_bls_coverage = {}
for key,val in soc2010_to_soc2018.items():
    if val is None:
        candidates = ambiguous_soc2010_to_soc2018[key]
        no_bls_coverage[key] = candidates
        
        print(f"[WARNING] Must be resolved manually for SOC 2010 Code: {key}")
        print(f"[INFO] Corresponding candidates: {candidates}\n")

[WARNING] Must be resolved manually for SOC 2010 Code: 13-2021
[INFO] Corresponding candidates: ['13-2022', '13-2023']

[WARNING] Must be resolved manually for SOC 2010 Code: 25-2052
[INFO] Corresponding candidates: ['25-2055', '25-2056']

[WARNING] Must be resolved manually for SOC 2010 Code: 25-9041
[INFO] Corresponding candidates: ['25-9042', '25-9043', '25-9049']

[WARNING] Must be resolved manually for SOC 2010 Code: 53-1031
[INFO] Corresponding candidates: ['53-1043', '53-1044', '53-1049']



In [17]:
# Check if any job titles in candidates exactly match the parent
resolved = []
for soc2010,candidates in no_bls_coverage.items():
    mask = df_soc2010_soc2018["2010 SOC Code"] == soc2010
    soc2010_jobtitle = (df_soc2010_soc2018[mask]["2010 SOC Title"]
                        .unique()[0]
                        .replace("(#)", "")
                        .strip())

    for c in candidates:
        mask = df_soc2010_soc2018["2018 SOC Code"] == c
        c_jobtitle = (df_soc2010_soc2018[mask]["2018 SOC Title"]
                      .unique()[0]
                      .replace("(#)", "")
                      .strip())
        if c_jobtitle == soc2010_jobtitle:
            soc2010_to_soc2018[soc2010] = c
            print(f"[INFO] Was able to resolve 2010 Code {soc2010} with",
                  "2018 code {c} through job title matching")
            resolved.append(soc2010)
            break

# Remove resolved items from dictionary
for code in resolved:
    del no_bls_coverage[code]

# Print info on items that must be manually resolved
for code, candidates in no_bls_coverage.items():
    print(f"[WARNING] Must be resolved manually for SOC 2010 Code: {code}")
    print(f"[INFO] Corresponding candidates: {candidates}\n")

[INFO] Was able to resolve 2010 Code 13-2021 with 2018 code {c} through job title matching
[WARNING] Must be resolved manually for SOC 2010 Code: 25-2052
[INFO] Corresponding candidates: ['25-2055', '25-2056']

[WARNING] Must be resolved manually for SOC 2010 Code: 25-9041
[INFO] Corresponding candidates: ['25-9042', '25-9043', '25-9049']

[WARNING] Must be resolved manually for SOC 2010 Code: 53-1031
[INFO] Corresponding candidates: ['53-1043', '53-1044', '53-1049']



In [18]:
# Manually hardcode/override for the keys whose candidates don't have BLS coverage
soc2010_to_soc2018["25-2052"] = "25-2056"
soc2010_to_soc2018["25-9041"] = "25-9042"
soc2010_to_soc2018["53-1031"] = "53-1043"

### Note
In the cells below, we add SOC 2010 codes in Census 2010 to SOC 2010 crosswalk that are not in SOC 2010 to SOC 2018. 

This is because according to BLS, the SOC 2010 to SOC 2018 crosswalk only publishes codes which have changed between iterations (i.e. if SOC 2010 is not in the crosswalk, that means that SOC 2018 is exactly the same).

Therefore, for each SOC 2010 code missing in the crosswalk, we find the closest/best SOC 2010 codes in the SOC 2010 to SOC 2018 crosswalk.
- Another thing: I did consider whether, since the SOC 2010 to SOC 2018 codes did not change throughout iterations, if there was a direct mapping between any of the SOC 2010 code and the O*NET-SOC 2019 code, but there was not.

In [19]:
def get_soc2010_prefix_length(soc2010):
    '''Prefix of SOC 2010 code is part that is not continuous zeroes'''
    digits = soc2010.replace("-", "")
    last_nonzero_idx = max(i for i,d in enumerate(digits) if d != "0")
    return last_nonzero_idx + 1

def get_children(soc2010_code, df_soc2010_soc2018):
    '''Determines which SOC 2010 codes in df_soc2010_soc2018 
       are similar to the target soc2010_code'''
    prefix_length = get_soc2010_prefix_length(soc2010_code)
    prefix = soc2010_code.replace("-", "")[:prefix_length]
    children = set()

    for soc2018 in df_soc2010_soc2018["2010 SOC Code"].unique():
        cand_prefix = soc2018.replace("-", "")[:prefix_length]
        if cand_prefix == prefix:
            children.add(soc2018)

    return children

unresolved = {}
for codes in df_gss_soc["2010 SOC Code"].unique():
    # split just in case since census to SOC 2010 expand horiz
    codes = [item.strip() for item in codes.split(",")]
    for soc2010 in codes:
        if soc2010 not in soc2010_to_soc2018:
            candidates = get_children(soc2010, df_soc2010_soc2018)
            if len(candidates) == 1:
                related_soc2010 = candidates.pop()
                soc2010_to_soc2018[soc2010] = soc2010_to_soc2018[related_soc2010]
                print(f"[INFO] Resolved {soc2010}")
            else:
                unresolved[soc2010] = candidates

In [20]:
# Try and match by exact job title match
resolved = set()
for soc2010, candidates in unresolved.items():
    soc2010_mask = df_gss_soc["2010 SOC Code"] == soc2010
    title = df_gss_soc[soc2010_mask]["Occupation Title"].unique()[0]

    for c in candidates:
        c_mask = df_soc2010_soc2018["2010 SOC Code"] == c
        c_title = (df_soc2010_soc2018[c_mask]["2010 SOC Title"]
                   .unique()[0].replace("(#)", "")
                   .strip())
        if c_title == title:
            soc2010_to_soc2018[soc2010] = soc2010_to_soc2018[c]
            resolved.add(soc2010)
            print(f"[INFO] Resolved code {soc2010} with code {c} through job title matching")
            break

while resolved:
    soc2010 = resolved.pop()
    del unresolved[soc2010]

[INFO] Resolved code 41-2010 with code 41-2011 through job title matching


In [21]:
# Load BLS data for 2016
df_bls = pd.read_excel(crosswalk_dir + "bls_2016.xlsx")
print(f"[INFO] Columns are: {df_bls.columns}")
df_bls.head()

[INFO] Columns are: Index(['area', 'area_title', 'area_type', 'naics', 'naics_title', 'own_code',
       'occ code', 'occ title', 'group', 'tot_emp', 'emp_prse', 'jobs_1000',
       'loc_quotient', 'pct_total', 'h_mean', 'a_mean', 'mean_prse', 'h_pct10',
       'h_pct25', 'h_median', 'h_pct75', 'h_pct90', 'a_pct10', 'a_pct25',
       'a_median', 'a_pct75', 'a_pct90', 'annual', 'hourly'],
      dtype='str')


,area,area_title,area_type,naics,naics_title,own_code,occ code,occ title,group,tot_emp,...,h_median,h_pct75,h_pct90,a_pct10,a_pct25,a_median,a_pct75,a_pct90,annual,hourly
0,99,U.S.,1,000000,Cross-industry,1235.0,00-0000,All Occupations,total,140400040,...,17.81,28.92,45.45,19290,24140,37040,60150,94540,NaN,NaN
1,99,U.S.,1,000000,Cross-industry,1235.0,11-0000,Management Occupations,major,7090790,...,48.46,70.72,#,47330,68630,100790,147090,#,NaN,NaN
2,99,U.S.,1,000000,Cross-industry,1235.0,11-1000,Top Executives,minor,2465800,...,49.19,78.35,#,42810,65420,102320,162970,#,NaN,NaN
3,99,U.S.,1,000000,Cross-industry,1235.0,11-1010,Chief Executives,broad,223260,...,87.12,#,#,69780,114100,181210,#,#,NaN,NaN
4,99,U.S.,1,000000,Cross-industry,1235.0,11-1011,Chief Executives,detailed,223260,...,87.12,#,#,69780,114100,181210,#,#,NaN,NaN


In [22]:
# Filter out to only the columns of interest since df_bls is large
cols = ["occ title", "occ code", "tot_emp"]
df_bls = df_bls[cols]
df_bls["tot_emp"] = pd.to_numeric(df_bls["tot_emp"], errors="coerce").astype("Int64")

In [23]:
# Try to resolve using BLS data, choosing candidate with most total employed
for soc2010, candidates in unresolved.items():
    related_soc2010 = tiebreaker(candidates, df_bls, 
                                 occ_code = "occ code", 
                                 tot_emp="tot_emp")
    if related_soc2010 is None:
        continue
    
    soc2010_to_soc2018[soc2010] = soc2010_to_soc2018[related_soc2010]
    resolved.add(soc2010)
    print(f"[INFO] Resolved code {soc2010}",
          f"with code {related_soc2010} through BLS total employment data.")

while resolved:
    soc2010 = resolved.pop()
    del unresolved[soc2010]

[INFO] Resolved code 11-2020 with code 11-2022 through BLS total employment data.
[INFO] Resolved code 11-9030 with code 11-9032 through BLS total employment data.
[INFO] Resolved code 13-1030 with code 13-1031 through BLS total employment data.
[INFO] Resolved code 13-1070 with code 13-1071 through BLS total employment data.
[INFO] Resolved code 13-2070 with code 13-2072 through BLS total employment data.
[INFO] Resolved code 15-1150 with code 15-1151 through BLS total employment data.
[INFO] Resolved code 15-2090 with code 15-2099 through BLS total employment data.
[INFO] Resolved code 17-1010 with code 17-1011 through BLS total employment data.
[INFO] Resolved code 17-1020 with code 17-1022 through BLS total employment data.
[INFO] Resolved code 17-2070 with code 17-2071 through BLS total employment data.
[INFO] Resolved code 17-2110 with code 17-2112 through BLS total employment data.
[INFO] Resolved code 17-3010 with code 17-3011 through BLS total employment data.
[INFO] Resolved 

In [24]:
# Get codes that must be manually resolved
for key,candidates in unresolved.items():
    print(f"[WARNING] Must be resolved manually for SOC 2010 Code: {key}")
    print(f"[INFO] Corresponding candidates: {candidates}\n")

# Building SOC 2010 to O*NET 2019 Crosswalk

In [25]:
missing_soc2018 = set(soc2010_to_soc2018.values()) - set(soc2018_to_onet2019.keys())
if missing_soc2018:
    print("[WARNING] There are missing SOC 2018 Codes",
          "which must be fixed before we build a mapping",
          "between SOC 2010 and O*NET-SOC 2019")

soc2010_to_onet2019 = {}
for soc2010,soc2018 in soc2010_to_soc2018.items():
    onet2019 = soc2018_to_onet2019[soc2018]
    soc2010_to_onet2019[soc2010] = onet2019

# Building 2010 Census Code to 2010 SOC Mapping

In [26]:
df_gss_soc.tail(20)

,Occupation Title,2010 Census Code,2010 SOC Code
515,Bridge and lock tenders,9340,53-6011
516,Parking lot attendants,9350,53-6021
517,Automotive and watercraft service attendants,9360,53-6031
518,Transportation inspectors,9410,53-6051
519,"Transportation attendants, except flight atten...",9415,53-6061
520,Other transportation workers,9420,"53-6041, 53-6099"
521,Conveyor operators and tenders,9500,53-7011
522,Crane and tower operators,9510,53-7021
523,"Dredge, excavating, and loading machine operators",9520,53-7030
524,Hoist and winch operators,9560,53-7041


In [27]:
df_gss_soc["2010 Census Code"].duplicated(keep=False).sum()

np.int64(0)

In [28]:
df_gss_soc["2010 SOC Code"].duplicated(keep=False).sum()

np.int64(0)

### Observations: Building 2010 Census Code to 2010 SOC Mapping
So it looks like this DataFrame is horizontally expanded, where some 2010 Census Codes will be associated with multiple 2010 SOC codes, denoted by the CSV in the `2010 SOC Code` column.

I'll handle this in a similar way as above, where I will use exact job title matching and BLS data to determine which candidate is the most fitting for 2010 Census Codes with multiple 2010 SOC Code candidates.

In [29]:
# One-to-ones only
census2010_to_soc2010 = {census2010:soc2010 for census2010,soc2010 
                         in zip(df_gss_soc["2010 Census Code"],
                                df_gss_soc["2010 SOC Code"]) 
                            if len(soc2010.split(",")) == 1}

# One-to-many only
ambiguous_census2010_to_soc2010 = {census2010:soc2010.replace(" ", "").split(",") 
                                   for census2010,soc2010 
                                   in zip(df_gss_soc["2010 Census Code"],
                                          df_gss_soc["2010 SOC Code"]) 
                                   if len(soc2010.replace(" ", "").split(",")) > 1}

In [30]:
# Check that dictionaries are disjoint and fully covering 
overlap = set(census2010_to_soc2010.keys()) & set(ambiguous_census2010_to_soc2010.keys())
disjoint = (len(overlap) == 0)

union = set(census2010_to_soc2010.keys()) | set(ambiguous_census2010_to_soc2010.keys())
fully_covers = union == set(df_gss_soc["2010 Census Code"].unique())
disjoint_and_fully_covers = disjoint and fully_covers

if not disjoint_and_fully_covers:
    print("[WARNING] Dictionaries are not disjoint or don't have full coverage of all possible 2010 Census Codes")
else:
    print("[INFO] Dictionaries are disjoint and have full coverage of all possible 2010 Census Codes")

[INFO] Dictionaries are disjoint and have full coverage of all possible 2010 Census Codes


In [31]:
# Find any with exact job title matches
resolved = set()
for census2010, soc2010_cands in ambiguous_census2010_to_soc2010.items():
    census_mask = df_gss_soc["2010 Census Code"] == census2010
    census_title = (df_gss_soc[census_mask]["Occupation Title"].unique()[0])
    for c in soc2010_cands:
        c_mask = df_bls["occ code"] == c
        c_title = df_bls[c_mask]["occ title"].unique()[0]
        if c_title == census_title:
            census2010_to_soc2010[census2010] = c
            resolved.add(census2010)
            print(f"[INFO] Resolved {census2010} with {c} through exact job title matching")

while resolved:
    code = resolved.pop()
    del ambiguous_census2010_to_soc2010[code]

In [32]:
# Use BLS employment data to choose candidate with the most employed
for census2010, soc2010_cands in ambiguous_census2010_to_soc2010.items():
    soc2010 = tiebreaker(soc2010_cands, df_bls, 
                         occ_code="occ code", 
                         tot_emp="tot_emp")
    if soc2010 is None:
        continue
    census2010_to_soc2010[census2010] = soc2010
    resolved.add(census2010)
    print(f"[INFO] Resolved {census2010} with {soc2010} through BLS employment data")

while resolved:
    code = resolved.pop()
    del ambiguous_census2010_to_soc2010[code]

[INFO] Resolved 1020 with 15-1132 through BLS employment data
[INFO] Resolved 2025 with 21-1099 through BLS employment data
[INFO] Resolved 2550 with 25-9031 through BLS employment data
[INFO] Resolved 3655 with 31-9099 through BLS employment data
[INFO] Resolved 3955 with 33-9099 through BLS employment data
[INFO] Resolved 4220 with 37-2011 through BLS employment data
[INFO] Resolved 4460 with 39-4021 through BLS employment data
[INFO] Resolved 6940 with 47-5099 through BLS employment data
[INFO] Resolved 7100 with 49-2094 through BLS employment data
[INFO] Resolved 7330 with 49-9041 through BLS employment data
[INFO] Resolved 7630 with 49-9099 through BLS employment data
[INFO] Resolved 9260 with 53-4041 through BLS employment data
[INFO] Resolved 9420 with 53-6099 through BLS employment data


In [33]:
# Get codes that need to be manually resolved
for key,val in ambiguous_census2010_to_soc2010.items():
    print(f"[WARNING] Need to resolve census code {key}")
    print(f"[WARNING] Candidates are: {val}")
    print()

# Building 2010 Census Code to O*NET-SOC 2019 Crosswalk

In [34]:
set(census2010_to_soc2010.values()) - set(soc2010_to_onet2019.keys())

set()

In [35]:
census2010_to_onet2019 = {}
for census2010, soc2010 in census2010_to_soc2010.items():
    onet_2019 = soc2010_to_onet2019[soc2010]
    census2010_to_onet2019[census2010] = onet_2019

census2010_to_onet2019

{10: '11-1011.00',
 20: '11-1021.00',
 30: '11-1031.00',
 40: '11-2011.00',
 50: '11-2022.00',
 60: '11-2032.00',
 100: '11-3012.00',
 110: '11-3021.00',
 120: '11-3031.00',
 135: '11-3111.00',
 136: '11-3121.00',
 137: '11-3131.00',
 140: '11-3051.00',
 150: '11-3061.00',
 160: '11-3071.00',
 205: '11-9013.00',
 220: '11-9021.00',
 230: '11-9032.00',
 300: '11-9041.00',
 310: '11-9051.00',
 325: '11-9171.00',
 330: '11-9071.00',
 340: '11-9081.00',
 350: '11-9111.00',
 360: '11-9121.00',
 400: '11-9131.00',
 410: '11-9141.00',
 420: '11-9151.00',
 425: '11-9161.00',
 430: '13-1082.00',
 500: '13-1011.00',
 510: '13-1021.00',
 520: '13-1022.00',
 530: '13-1023.00',
 540: '13-1031.00',
 565: '13-1041.00',
 600: '13-1051.00',
 630: '13-1071.00',
 640: '13-1141.00',
 650: '13-1151.00',
 700: '13-1081.00',
 710: '13-1111.00',
 725: '13-1121.00',
 726: '13-1131.00',
 735: '13-1161.00',
 740: '13-1199.00',
 800: '13-2011.00',
 810: '13-2023.00',
 820: '13-2031.00',
 830: '13-2041.00',
 840: 

In [36]:
# Write our CSV
with open(crosswalk_dir + "census2010_to_onet2019.csv", mode="w", newline="") as file:
    writer = csv.writer(file)

    # CSV header
    writer.writerow(["2010 Census Code", 
                     "2010 Census Title", 
                     "O*NET-SOC 2019 Code", 
                     "O*NET-SOC 2019 Title"])
    
    # Write each key-value pair as a row
    for census2010, onet2019 in census2010_to_onet2019.items():
        census_mask = df_gss_soc["2010 Census Code"] == census2010
        census_title = df_gss_soc[census_mask]["Occupation Title"].unique()[0]

        onet_mask = df_onet2019_soc2018["O*NET-SOC 2019 Code"] == onet2019
        onet_title = df_onet2019_soc2018[onet_mask]["O*NET-SOC 2019 Title"].unique()[0]
        writer.writerow([census2010, census_title, onet2019, onet_title])

# ISCO 2008 to SOC 2010 mapping

In [37]:
df_isco_soc2010 = pd.read_csv(crosswalk_dir + "/ISCO_SOC_Crosswalk.csv", 
                              dtype={"ISCO-08 Code": str})
df_isco_soc2010 = df_isco_soc2010.astype(str)
df_isco_soc2010.head()

,ISCO-08 Code,ISCO-08 Title EN,part,2010 SOC Code,2010 SOC Title,Comment 8/17/11
0,0110,Commissioned armed forces officers,*,55-1011,Air Crew Officers,NaN
1,0110,Commissioned armed forces officers,*,55-1012,Aircraft Launch and Recovery Officers,NaN
2,0110,Commissioned armed forces officers,*,55-1013,Armored Assault Vehicle Officers,NaN
3,0110,Commissioned armed forces officers,*,55-1014,Artillery and Missile Officers,NaN
4,0110,Commissioned armed forces officers,*,55-1015,Command and Control Center Officers,NaN


In [38]:
df_isco_soc2010.drop(columns="Comment 8/17/11", inplace=True)
df_isco_soc2010.head()

,ISCO-08 Code,ISCO-08 Title EN,part,2010 SOC Code,2010 SOC Title
0,0110,Commissioned armed forces officers,*,55-1011,Air Crew Officers
1,0110,Commissioned armed forces officers,*,55-1012,Aircraft Launch and Recovery Officers
2,0110,Commissioned armed forces officers,*,55-1013,Armored Assault Vehicle Officers
3,0110,Commissioned armed forces officers,*,55-1014,Artillery and Missile Officers
4,0110,Commissioned armed forces officers,*,55-1015,Command and Control Center Officers


In [39]:
df_isco_soc2010["2010 SOC Code"] = df_isco_soc2010["2010 SOC Code"].str.strip()

In [40]:
# Split into ambiguous and unambiguous data frames
ambiguous_mask = df_isco_soc2010["ISCO-08 Code"].duplicated(keep=False)
df_isco_ambiguous = df_isco_soc2010[ambiguous_mask]
df_isco_unambiguous = df_isco_soc2010[~ambiguous_mask]

# Construct unambiguous dictionary
isco_to_soc2010 = {isco:soc for isco,soc 
                   in zip(df_isco_unambiguous["ISCO-08 Code"], 
                          df_isco_unambiguous["2010 SOC Code"])}

# Construct ambiguous dictionary
ambiguous_isco_to_soc2010 = {}
for idx, row in df_isco_ambiguous.iterrows():
    isco = row["ISCO-08 Code"]
    soc = row["2010 SOC Code"]
    ambiguous_isco_to_soc2010.setdefault(isco, []).append(soc)

In [41]:
# Try to resolve with exact job title match
resolved = set()
for isco, soc_cands in ambiguous_isco_to_soc2010.items():
    isco_mask = df_isco_ambiguous["ISCO-08 Code"] == isco
    isco_title = df_isco_ambiguous[isco_mask]["ISCO-08 Title EN"].unique()[0]

    for soc in soc_cands:
        soc_mask = df_isco_ambiguous["2010 SOC Code"] == soc
        soc_title = df_isco_ambiguous[soc_mask]["2010 SOC Title"].unique()[0]

        if isco_title == soc_title:
            isco_to_soc2010[isco] = soc
            resolved.add(isco)
            print(f"[INFO] Resolved {isco} with {soc} using job title matching")
            break

while resolved:
    isco = resolved.pop()
    del ambiguous_isco_to_soc2010[isco]

[INFO] Resolved 2113 with 19-2031 using job title matching


In [42]:
# Use BLS data to get the best candidate that maximizes coverage
for isco, soc_cands in ambiguous_isco_to_soc2010.items():
    soc2010 = tiebreaker(soc_cands, df_bls, occ_code="occ code", tot_emp="tot_emp")
    if soc2010 is None:
        continue
    isco_to_soc2010[isco] = soc2010
    resolved.add(isco)
    print(f"[INFO] Resolved {isco} with {soc2010} using BLS total employed data")

while resolved:
    isco = resolved.pop()
    del ambiguous_isco_to_soc2010[isco]

[INFO] Resolved 1112 with 11-1021 using BLS total employed data
[INFO] Resolved 1113 with 11-1011 using BLS total employed data
[INFO] Resolved 1114 with 11-1021 using BLS total employed data
[INFO] Resolved 1120 with 11-1021 using BLS total employed data
[INFO] Resolved 1212 with 11-3121 using BLS total employed data
[INFO] Resolved 1219 with 11-9199 using BLS total employed data
[INFO] Resolved 1221 with 11-2022 using BLS total employed data
[INFO] Resolved 1222 with 11-2031 using BLS total employed data
[INFO] Resolved 1223 with 11-9041 using BLS total employed data
[INFO] Resolved 1343 with 11-1021 using BLS total employed data
[INFO] Resolved 1345 with 11-9032 using BLS total employed data
[INFO] Resolved 1346 with 11-1021 using BLS total employed data
[INFO] Resolved 1431 with 11-9199 using BLS total employed data
[INFO] Resolved 2111 with 19-2012 using BLS total employed data
[INFO] Resolved 2114 with 19-2042 using BLS total employed data
[INFO] Resolved 2120 with 15-2031 using 

In [43]:
# Print warnings
for key,val in ambiguous_isco_to_soc2010.items():
    print(f"[WARNING] Must resolve ISCO Code {key} manually")
    print(f"[WARNING] Candidates are: {val}")
    print()

[WARNING] Must resolve ISCO Code 0110 manually
[WARNING] Candidates are: ['55-1011', '55-1012', '55-1013', '55-1014', '55-1015', '55-1016', '55-1017', '55-1019']

[WARNING] Must resolve ISCO Code 0210 manually
[WARNING] Candidates are: ['55-2011', '55-2012', '55-2013']

[WARNING] Must resolve ISCO Code 0310 manually
[WARNING] Candidates are: ['55-3011', '55-3012', '55-3013', '55-3014', '55-3015', '55-3016', '55-3017', '55-3018', '55-3019']



In [44]:
# Manually resolve
isco_to_soc2010["0110"] = "55-1016"
isco_to_soc2010["0210"] = "55-2013"
isco_to_soc2010["0310"] = "55-3016"

# Building ISCO 2008 to O*NET-SOC 2019 Crosswalk

In [45]:
# Construct our map
isco_to_onet2019 = {}
for isco, soc2010 in isco_to_soc2010.items():
    soc2018 = soc2010_to_soc2018[soc2010]
    onet_2019 = soc2018_to_onet2019[soc2018]
    isco_to_onet2019[isco] = onet_2019

isco_to_onet2019

{'1111': '11-1031.00',
 '1211': '11-3031.00',
 '1213': '13-1082.00',
 '1311': '11-9013.00',
 '1312': '11-9013.00',
 '1321': '11-3051.00',
 '1322': '13-1082.00',
 '1323': '11-9021.00',
 '1324': '11-3071.00',
 '1330': '11-3021.00',
 '1341': '11-9031.00',
 '1342': '11-9111.00',
 '1344': '11-9151.00',
 '1349': '13-1082.00',
 '1411': '11-9081.00',
 '1412': '11-9051.00',
 '1420': '11-1021.00',
 '1439': '13-1082.00',
 '211': '19-2099.00',
 '2112': '19-2021.00',
 '2141': '17-2112.00',
 '2142': '17-2051.00',
 '2143': '17-2081.00',
 '2145': '17-2041.00',
 '2151': '17-2071.00',
 '2153': '17-2072.00',
 '2161': '17-1011.00',
 '2162': '17-1012.00',
 '2164': '19-3051.00',
 '2222': '29-1161.00',
 '2230': '29-1299.00',
 '2240': '29-1071.00',
 '2250': '29-1131.00',
 '2262': '29-1051.00',
 '2265': '29-1031.00',
 '2267': '29-1041.00',
 '2330': '25-2031.00',
 '2351': '25-9031.00',
 '2354': '25-3021.00',
 '2356': '13-1151.00',
 '2422': '13-1199.00',
 '2424': '13-1151.00',
 '2432': '27-3031.00',
 '2513': '15

In [46]:
# Write our CSV
with open(crosswalk_dir + "isco2008_to_onet2019.csv", mode="w", newline="") as file:
    writer = csv.writer(file)

    # CSV header
    writer.writerow(["ISCO-08 Code", 
                     "ISCO-08 Title EN", 
                     "O*NET-SOC 2019 Code", 
                     "O*NET-SOC 2019 Title"])
    
    # Write each key-value pair as a row
    for isco, onet2019 in isco_to_onet2019.items():
        isco_mask = df_isco_soc2010["ISCO-08 Code"] == isco
        isco_title = df_isco_soc2010[isco_mask]["ISCO-08 Title EN"].unique()[0]

        onet_mask = df_onet2019_soc2018["O*NET-SOC 2019 Code"] == onet2019
        onet_title = df_onet2019_soc2018[onet_mask]["O*NET-SOC 2019 Title"].unique()[0]
        writer.writerow([isco, isco_title, onet2019, onet_title])